"""
realtime_prediction.py

This module performs real-time American Sign Language recognition.

Workflow:
1. Capture live video from the webcam.
2. Detect the hand using MediaPipe.
3. Extract and normalize landmarks.
4. Load the trained classification model.
5. Predict the performed ASL gesture.
6. Display the prediction on the video stream.

This module represents the final deployment stage of the project.
"""

In [19]:
import cv2
import numpy as np
import mediapipe as mp

In [20]:
import joblib

linear_svm_model = joblib.load(
    "../models/linear_svm_raw.pkl"
)

raw_scaler = joblib.load(
    "../models/raw_scaler.pkl"
)

rbf_svm = joblib.load(
    "../models/rbf_svc.pkl"
)
raw_linear_svm = joblib.load(
    "../models/raw_linear_svm.pkl"
)

In [21]:
current_model = raw_linear_svm
current_model_name = "Wrist Centered Linear SVM"

In [22]:
# Camera settings
WIDTH = 640
HEIGHT = 480

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, HEIGHT)

# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    model_complexity=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.5
)

In [23]:
class_names = [
    "A", "B", "C", "D", "E", "F", "G", "H", "I", "J",
    "K", "L", "M", "N", "O", "P", "Q", "R", "S", "T",
    "U", "V", "W", "X", "Y", "Z", "del", "space"
]

In [24]:
def extract_landmarks(hand_landmarks):

    landmarks = []

    for landmark in hand_landmarks.landmark:

        landmarks.extend([
            landmark.x,
            landmark.y,
            landmark.z
        ])

    return np.array(landmarks).reshape(1, -1)

In [25]:
def predict_gesture(X, scaler, model, class_names):

    X_scaled = scaler.transform(X)

    prediction = model.predict(X_scaled)

    predicted_class = prediction[0]

    label = class_names[predicted_class]

    probabilities = model.predict_proba(X_scaled)

    confidence = np.max(probabilities)

    return label, confidence

In [26]:
from collections import deque, Counter

prediction_history = deque(maxlen=10)

sentence = ""
last_added_letter = None



In [27]:
from collections import deque, Counter
prediction_history = deque(maxlen=10)

while True:

    success, frame = cap.read()

    if not success:
        print("Error: Can't receive frame")
        break

    # =====================================
    # Default prediction
    # =====================================

    label = "No hand"
    confidence = 0.0
    stable_label = "No hand"

    # =====================================
    # 1. ML PROCESSING — ORIGINAL FRAME
    # =====================================

    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    results = hands.process(rgb_frame)

    if results.multi_hand_landmarks:

        # Currently using one hand
        hand_landmarks = results.multi_hand_landmarks[0]

        # ---------------------------------
        # Draw landmarks on ORIGINAL frame
        # ---------------------------------

        mp_draw.draw_landmarks(
            frame,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS,
            mp_draw.DrawingSpec(
                color=(0, 255, 0),
                thickness=4,
                circle_radius=5
            ),
            mp_draw.DrawingSpec(
                color=(0, 0, 255),
                thickness=4
            )
        )

        # ---------------------------------
        # Extract landmarks
        # ---------------------------------

        X = extract_landmarks(hand_landmarks)

        # ---------------------------------
        # Prediction
        # ---------------------------------

        label, confidence = predict_gesture(
            X,
            raw_scaler,
            current_model,
            class_names
        )

        # Add current prediction to history
        prediction_history.append(label)

        # Majority vote
        stable_label = Counter(
        prediction_history
        ).most_common(1)[0][0]

    else:
        prediction_history.clear()
        # Allow the same letter to be entered again
        last_added_letter = None

    # =====================================
    # 2. VISUALIZATION
    # =====================================

    # Flip ONLY for visualization
    display_frame = cv2.flip(frame, 1)

    # ---------------------------------
    # Prediction
    # ---------------------------------

    cv2.putText(
        display_frame,
        f"Prediction: {label}",
        (20, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (0, 255, 0),
        3
    )

    # ---------------------------------
    # Confidence
    # ---------------------------------

    cv2.putText(
        display_frame,
        f"Confidence: {confidence * 100:.2f}%",
        (20, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0, 255, 0),
        2
    )

    # ---------------------------------
    # Current model
    # ---------------------------------

    cv2.putText(
        display_frame,
        f"Model: {current_model_name}",
        (20, 130),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 255, 0),
        1
    )

    # ---------------------------------
    # Controls
    # ---------------------------------

    cv2.putText(
        display_frame,
        "1: Linear SVM | 2: RBF SVM |3: Raw_Linear SVM | Q: Quit",
        (20, 460),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        1
    )

    # =====================================
    # 3. DISPLAY
    # =====================================

    cv2.imshow(
        "Hand Gesture Recognition",
        display_frame
    )

    # =====================================
    # 4. KEYBOARD CONTROL
    # =====================================

    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        break

    elif key == ord('1'):

        current_model = linear_svm_model
        current_model_name = "Linear SVM"

    elif key == ord('2'):

        current_model = rbf_svm
        current_model_name = "RBF SVM"

    elif key == ord('3'):
    
            current_model = raw_linear_svm
            current_model_name = "Wrist Centered Linear SVM"

cap.release()
cv2.destroyAllWindows()

Error: Can't receive frame
